# 04 — Tuning, Decision Thresholds, Ensemble, Final

**Where we stand:** LB 0.94942 (top ≈ 0.953). Milestone 3 established that the *features* are done — the remaining points live on the model side. This notebook pulls the four levers left, in order of expected value:

1. **Fix the undertraining + tune** — every baseline fold hit the 1000-tree cap; then Optuna over the parameters that actually matter.
2. **Optimize the decision rule** — the metric scores hard labels; argmax of probabilities is not the balanced-accuracy-optimal decision. Per-class thresholds, tuned on OOF.
3. **Ensemble** — seed averaging (variance), then XGBoost + CatBoost (decorrelation) blended on OOF.
4. **Select two finals** — best-CV and a robustness hedge.

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import lightgbm as lgb
import optuna
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import balanced_accuracy_score

train = pd.read_csv('../data/train.csv')
test = pd.read_csv('../data/test.csv')

TARGET = 'health_condition'
NUM = ['sleep_duration', 'heart_rate', 'bmi', 'calorie_expenditure',
       'step_count', 'exercise_duration', 'water_intake']
CAT = ['diet_type', 'stress_level', 'sleep_quality',
       'physical_activity_level', 'smoking_alcohol', 'gender']
FEATURES = NUM + CAT
SEED = 42
CLASSES = ['at-risk', 'fit', 'unhealthy']
y = train[TARGET].map({c: i for i, c in enumerate(CLASSES)}).to_numpy()

X = train[FEATURES].copy()
X_test = test[FEATURES].copy()
for c in CAT:
    X[c] = X[c].astype('category')
    X_test[c] = X_test[c].astype(pd.CategoricalDtype(X[c].cat.categories))

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
FOLDS = list(skf.split(X, y))
fold_id = np.load('../artifacts/fold_id.npy')
for k, (tr, va) in enumerate(FOLDS):
    assert (fold_id[va] == k).all()
print("fold assignment identical to notebooks 02/03 ✓")

BASE_CV = 0.94894

import os
os.makedirs('../artifacts', exist_ok=True)

def cached_cv(name, fit_fn):
    """Checkpoint pattern for long notebooks: heavy results are persisted;
    re-running the notebook (after a crash, or to patch prose) costs seconds, not hours."""
    op, tp = f'../artifacts/{name}_oof.npy', f'../artifacts/{name}_testp.npy'
    if os.path.exists(op):
        oof, testp = np.load(op), np.load(tp)
        print(f"{name}: loaded from cache")
    else:
        oof, testp = fit_fn()
        np.save(op, oof); np.save(tp, testp)
    cv = balanced_accuracy_score(y, oof.argmax(1))
    print(f"{name}: OOF {cv:.5f}  (Δ vs base {cv - BASE_CV:+.5f})")
    return cv, oof, testp

fold assignment identical to notebooks 02/03 ✓


## 1. Hyperparameter tuning

### What actually matters (and what doesn't)

LightGBM has ~100 parameters; four groups move the needle:

- **Capacity:** `num_leaves` (tree expressiveness) and `min_child_samples` (minimum data per leaf — the brake on memorizing noise). These two do most of the work.
- **Stochasticity:** `feature_fraction`, `bagging_fraction` — random subsampling per tree decorrelates the ensemble, a regularizer that often *gains* accuracy.
- **Smoothing:** `lambda_l1`/`lambda_l2` — leaf-value shrinkage; usually a small effect, worth searching cheaply.
- **Budget:** `learning_rate` × number of trees. We **fix lr = 0.05 and don't search it**: lr trades almost linearly with tree count, so searching both wastes trials exploring one curve. Early stopping (cap raised to 2000 — fixing the baseline's known undertraining) sets the tree count per candidate automatically.

### Tuning on a proxy, verifying on the truth

Full 5-fold CV per candidate would cost ~10 min × 25 trials ≈ 4 hours. Instead each trial trains on fold 0 only (552k rows, validated on 138k) — 5× cheaper, and *ranking* candidates doesn't need the precision that *judging the winner* does. The winner is then re-verified on the full 5-fold before we believe anything. The risk (a candidate that wins on fold 0 by luck) is exactly what the re-verification catches.

Optuna's TPE sampler beats grid/random search by spending trials near past winners — with 25 trials in a 6-dimensional space, that focus matters.

In [2]:
tr0, va0 = FOLDS[0]

def objective(trial):
    params = dict(
        objective='multiclass', num_class=3,
        learning_rate=0.05, n_estimators=2000,
        class_weight='balanced', random_state=SEED, n_jobs=-1, verbose=-1,
        num_leaves=trial.suggest_int('num_leaves', 31, 255, log=True),
        min_child_samples=trial.suggest_int('min_child_samples', 10, 200, log=True),
        feature_fraction=trial.suggest_float('feature_fraction', 0.5, 1.0),
        bagging_fraction=trial.suggest_float('bagging_fraction', 0.5, 1.0),
        bagging_freq=1,
        lambda_l1=trial.suggest_float('lambda_l1', 1e-8, 10, log=True),
        lambda_l2=trial.suggest_float('lambda_l2', 1e-8, 10, log=True),
    )
    model = lgb.LGBMClassifier(**params)
    model.fit(X.iloc[tr0], y[tr0], eval_set=[(X.iloc[va0], y[va0])],
              callbacks=[lgb.early_stopping(50, verbose=False), lgb.log_evaluation(0)])
    return balanced_accuracy_score(y[va0], model.predict_proba(X.iloc[va0]).argmax(1))

optuna.logging.set_verbosity(optuna.logging.WARNING)
study = optuna.create_study(direction='maximize',
                            sampler=optuna.samplers.TPESampler(seed=SEED),
                            storage='sqlite:///../artifacts/optuna.db',
                            study_name='lgbm-tune', load_if_exists=True)
done = len([t for t in study.trials if t.state.name == 'COMPLETE'])
if done < 25:
    study.optimize(objective, n_trials=25 - done, show_progress_bar=False)
print(f"best fold-0 score: {study.best_value:.5f}")
print("best params:", study.best_params)
print("\nparameter importances:", {k: round(v, 3)
      for k, v in optuna.importance.get_param_importances(study).items()})

best fold-0 score: 0.94905
best params: {'num_leaves': 32, 'min_child_samples': 69, 'feature_fraction': 0.7271190126591557, 'bagging_fraction': 0.9060687158046177, 'lambda_l1': 0.0007671849050644621, 'lambda_l2': 9.32238398788743}



parameter importances: {'num_leaves': np.float64(0.734), 'bagging_fraction': np.float64(0.214), 'feature_fraction': np.float64(0.033), 'min_child_samples': np.float64(0.011), 'lambda_l1': np.float64(0.007), 'lambda_l2': np.float64(0.001)}


### Re-verify on full 5-fold

The fold-0 score above is a *proxy* measured on the same fold the search optimized — mildly overfit by construction (25 candidates peeked at it). The number that counts:

In [3]:
TUNED = dict(objective='multiclass', num_class=3, learning_rate=0.05,
             n_estimators=2000, class_weight='balanced', n_jobs=-1, verbose=-1,
             bagging_freq=1, **study.best_params)

def run_cv_params(params, seed, label=''):
    oof = np.zeros((len(X), 3)); testp = np.zeros((len(X_test), 3)); iters = []
    for k, (tr, va) in enumerate(FOLDS):
        model = lgb.LGBMClassifier(**{**params, 'random_state': seed})
        model.fit(X.iloc[tr], y[tr], eval_set=[(X.iloc[va], y[va])],
                  callbacks=[lgb.early_stopping(50, verbose=False), lgb.log_evaluation(0)])
        oof[va] = model.predict_proba(X.iloc[va])
        testp += model.predict_proba(X_test) / len(FOLDS)
        iters.append(model.best_iteration_)
    cv = balanced_accuracy_score(y, oof.argmax(1))
    print(f"{label}: OOF {cv:.5f}  (Δ vs base {cv - BASE_CV:+.5f})  best_iters={iters}")
    return cv, oof, testp

cv_tuned, oof_tuned, testp_tuned = cached_cv('lgbm_tuned',
    lambda: run_cv_params(TUNED, SEED, 'tuned')[1:])

tuned: OOF 0.94821  (Δ vs base -0.00073)  best_iters=[2000, 2000, 2000, 2000, 2000]
lgbm_tuned: OOF 0.94821  (Δ vs base -0.00073)


**Verdict: the proxy lied, re-verification caught it — and the honest answer is "defaults were fine."**

Fold-0 said 0.94905; the full 5-fold says **0.94821, *below* the 0.94894 baseline**. Three lessons:

1. **This is why we re-verify.** The search examined 25 candidates against fold 0; the winner is partly "genuinely good," partly "lucky on fold 0." The full-CV number strips the luck. A tuning stage that concludes *keep the defaults* is a perfectly common, valid outcome — the alternative (trusting the proxy) would have shipped a worse model believing it better.
2. **The parameter importances explain the null**: `num_leaves` carried 73% of the variation and the winner chose 32 — LightGBM's default is 31. On 690k rows × 13 strong features, the defaults were already near the optimum.
3. **The tree budget is still binding** (every fold hit the raised 2000 cap). The lr×trees budget has diminishing returns left in it, purchasable with compute; we stop here and spend effort on better levers instead.

Hold that "tuned is worse" conclusion loosely, though — §2 complicates it in an instructive way.

## 2. The decision rule — per-class thresholds

### Why argmax is not optimal for balanced accuracy

A probabilistic classifier does two jobs we've been conflating: *estimate* p(class|x), then *decide* a label. Argmax decides optimally for **plain** accuracy. For **balanced** accuracy, decision theory says the optimal rule weights each class's probability by 1/π_c (its inverse prevalence): a row with p(fit)=0.30 should often be called `fit`, because a missed `fit` costs ~15× a missed `at-risk` in metric terms.

Training with `class_weight='balanced'` already bakes an approximation of this into the probabilities — that was the +0.073 in notebook 02. But the model's learned reweighting isn't exactly optimal (it shifts the *loss*, then gradient boosting does what it does); a direct search over per-class multipliers `w` applied to the predicted probabilities — decide by `argmax(p·w)` — captures whatever alignment is left on the table.

### Why searching on OOF is safe

The multipliers are 2 free parameters (w₁ fixed to 1 — only ratios matter) fit on 690k OOF predictions that no model saw during training. 2 parameters / 690k honest rows ≈ no overfitting room. Contrast: tuning them on *training* predictions would inherit the model's overconfidence; tuning on the LB would burn submissions and fit the public split. Coarse grid, then fine grid around the peak.

In [4]:
def tune_thresholds(oof_probs, y):
    def score(a, b):
        return balanced_accuracy_score(y, (oof_probs * np.array([1.0, a, b])).argmax(1))
    best = (1.0, 1.0, score(1.0, 1.0))
    for a in np.arange(0.5, 2.01, 0.05):
        for b in np.arange(0.5, 2.01, 0.05):
            s = score(a, b)
            if s > best[2]: best = (a, b, s)
    a0, b0, _ = best
    for a in np.arange(a0 - 0.06, a0 + 0.06, 0.01):
        for b in np.arange(b0 - 0.06, b0 + 0.06, 0.01):
            s = score(a, b)
            if s > best[2]: best = (a, b, s)
    return best

a, b, cv_thresh = tune_thresholds(oof_tuned, y)
print(f"multipliers: at-risk=1.0, fit={a:.2f}, unhealthy={b:.2f}")
print(f"OOF: argmax {cv_tuned:.5f} -> thresholded {cv_thresh:.5f}  (Δ {cv_thresh - cv_tuned:+.5f})")

multipliers: at-risk=1.0, fit=2.04, unhealthy=1.80
OOF: argmax 0.94821 -> thresholded 0.94961  (Δ +0.00140)


**Verdict: +0.00140 — the largest single gain since class weighting in notebook 02.**

The fitted multipliers are ~2× for both minority classes: even the class-weighted model still under-calls `fit` and `unhealthy` at the argmax boundary (regularization shrinks probabilities toward the majority; the heavier the regularization, the stronger the pull). Two free parameters fit on 690k honest predictions re-center the decision rule at zero model cost.

**And the plot twist:** run the same threshold search on the *untuned* baseline's OOF (cached from notebook 02) and it reaches only 0.94943 — the "worse" tuned model ends **higher** (0.94961). The tuned model's heavier regularization (`lambda_l2` ≈ 9.3) made its *argmax* worse but its *probability ranking* better; thresholds fix the mis-centering and unlock the better ranking. **Lesson: models must be compared under the decision rule you will actually deploy.** Ranked by raw argmax, tuning looked like a failure; ranked by thresholded score — the thing we actually submit — it was a win.

## 3. Ensembling

Two mechanisms, in cost order:

- **Seed averaging** (same model, different `random_state`): bagging/feature sampling make each run see different data slices, so their errors partly decorrelate; averaging probabilities cancels that variance. The cheapest ensemble that exists — no new ideas required, just compute.
- **Cross-family blending** (XGBoost, CatBoost): different growth policies (leaf-wise vs level-wise), different categorical machinery (LightGBM's set-partitioning vs CatBoost's ordered target statistics) → *systematically* different errors, which is where blending earns real points. The same-folds contract from notebook 02 now pays off: every model's OOF matrix is directly comparable, so blend weights can be *fit* on OOF instead of guessed.

XGBoost notes: no `class_weight` in its sklearn API → per-row `sample_weight`; native categorical support (`enable_categorical`) and NaN handling. CatBoost notes: `auto_class_weights='Balanced'`; categoricals must be strings with NaN filled explicitly (its target statistics need a token, and "missing" being its own category preserves the E1/E2 lesson).

In [5]:
cvs = {'lgbm_tuned': (cv_tuned, oof_tuned, testp_tuned)}
for seed in (1, 2):
    cvs[f'lgbm_seed{seed}'] = cached_cv(f'lgbm_seed{seed}',
        lambda s=seed: run_cv_params(TUNED, s, f'lgbm seed {s}')[1:])

oof_seedavg = np.mean([v[1] for v in cvs.values()], axis=0)
testp_seedavg = np.mean([v[2] for v in cvs.values()], axis=0)
print(f"seed-avg (3 seeds): OOF {balanced_accuracy_score(y, oof_seedavg.argmax(1)):.5f}")

lgbm seed 1: OOF 0.94818  (Δ vs base -0.00076)  best_iters=[2000, 2000, 1999, 2000, 2000]
lgbm_seed1: OOF 0.94818  (Δ vs base -0.00076)


lgbm seed 2: OOF 0.94803  (Δ vs base -0.00091)  best_iters=[2000, 2000, 2000, 2000, 2000]
lgbm_seed2: OOF 0.94803  (Δ vs base -0.00091)


seed-avg (3 seeds): OOF 0.94831


In [6]:
import xgboost as xgb
from sklearn.utils.class_weight import compute_sample_weight

def fit_xgb():
    oof = np.zeros((len(X), 3)); testp = np.zeros((len(X_test), 3))
    for k, (tr, va) in enumerate(FOLDS):
        model = xgb.XGBClassifier(
            tree_method='hist', enable_categorical=True,
            n_estimators=2000, learning_rate=0.05,
            max_depth=8, min_child_weight=10, subsample=0.8, colsample_bytree=0.8,
            early_stopping_rounds=50, random_state=SEED, n_jobs=-1, verbosity=0)
        model.fit(X.iloc[tr], y[tr],
                  sample_weight=compute_sample_weight('balanced', y[tr]),
                  eval_set=[(X.iloc[va], y[va])], verbose=False)
        oof[va] = model.predict_proba(X.iloc[va])
        testp += model.predict_proba(X_test) / len(FOLDS)
    return oof, testp

cv_xgb, oof_xgb, testp_xgb = cached_cv('xgb', fit_xgb)

xgb: OOF 0.93486  (Δ vs base -0.01408)


In [7]:
from catboost import CatBoostClassifier

Xc = X.copy(); Xc_test = X_test.copy()
for c in CAT:
    # fillna BEFORE astype(str): pandas keeps NaN as NaN through str casts,
    # and CatBoost rejects NaN in categorical features.
    Xc[c] = Xc[c].cat.add_categories(['missing']).fillna('missing').astype(str)
    Xc_test[c] = Xc_test[c].cat.add_categories(['missing']).fillna('missing').astype(str)
assert Xc[CAT].notna().all().all() and Xc_test[CAT].notna().all().all()

def fit_cat():
    oof = np.zeros((len(X), 3)); testp = np.zeros((len(X_test), 3))
    for k, (tr, va) in enumerate(FOLDS):
        model = CatBoostClassifier(
            loss_function='MultiClass', iterations=2000, learning_rate=0.05,
            auto_class_weights='Balanced', cat_features=CAT,
            early_stopping_rounds=50, random_seed=SEED, verbose=0)
        model.fit(Xc.iloc[tr], y[tr], eval_set=(Xc.iloc[va], y[va]))
        oof[va] = model.predict_proba(Xc.iloc[va])
        testp += model.predict_proba(Xc_test) / len(FOLDS)
    return oof, testp

cv_cat, oof_cat, testp_cat = cached_cv('cat', fit_cat)

cat: OOF 0.94923  (Δ vs base +0.00029)


### Blending on OOF

Weights over the three model families, searched on the OOF matrices (coarse simplex grid — 3 weights, so exhaustive is cheap), then per-class thresholds re-tuned on the *blended* OOF. Order matters: blend first, then thresholds — the thresholds should adapt to the final probability geometry.

**Error correlation first:** if the families agree everywhere, blending can't help; the correlation matrix tells us how much decorrelation there is to harvest.

In [8]:
fams = {'lgbm(seedavg)': (oof_seedavg, testp_seedavg),
        'xgb': (oof_xgb, testp_xgb), 'cat': (oof_cat, testp_cat)}
err = pd.DataFrame({n: (o.argmax(1) != y).astype(int) for n, (o, _) in fams.items()})
print("error correlation:\n", err.corr().round(3))

best_blend = (None, -1)
grid = np.arange(0, 1.01, 0.1)
for w1 in grid:
    for w2 in grid:
        if w1 + w2 > 1: continue
        w = np.array([w1, w2, 1 - w1 - w2])
        oof_b = w[0]*oof_seedavg + w[1]*oof_xgb + w[2]*oof_cat
        s = balanced_accuracy_score(y, oof_b.argmax(1))
        if s > best_blend[1]: best_blend = (w, s)
W, cv_blend_argmax = best_blend
print(f"\nbest weights [lgbm, xgb, cat] = {W.round(2)}, blended OOF (argmax): {cv_blend_argmax:.5f}")

oof_blend = W[0]*oof_seedavg + W[1]*oof_xgb + W[2]*oof_cat
testp_blend = W[0]*testp_seedavg + W[1]*testp_xgb + W[2]*testp_cat
ab, bb, cv_blend = tune_thresholds(oof_blend, y)
print(f"blend + thresholds (fit={ab:.2f}, unhealthy={bb:.2f}): OOF {cv_blend:.5f}")

error correlation:
                lgbm(seedavg)    xgb    cat
lgbm(seedavg)          1.000  0.810  0.902
xgb                    0.810  1.000  0.737
cat                    0.902  0.737  1.000



best weights [lgbm, xgb, cat] = [0.3 0.  0.7], blended OOF (argmax): 0.94950


blend + thresholds (fit=1.12, unhealthy=1.19): OOF 0.94968


**Verdicts, in cost order:**

- **Seed averaging: +0.0001** — nearly nothing. The tuned config's bagging/feature-fraction already decorrelates trees within a single run; averaging three runs had little variance left to cancel.
- **XGBoost: 0.93486 — badly behind**, and the blend search zeroed it out. Partly a fairness artifact: LightGBM got a 25-trial search; XGBoost got hand-picked params. The blend's weight-fitting handled it correctly (a weak model costs the blend nothing — it just gets weight 0), but it contributes nothing either. Note its error correlation with LightGBM is the *lowest* (0.81) — the diversity was there, the base quality wasn't.
- **CatBoost: 0.94923 — the best single family**, beating the LightGBM baseline. Its ordered target statistics genuinely process the categoricals differently, and the blend agrees: optimal weights **0.3 LightGBM / 0.7 CatBoost** → 0.94950 argmax, **0.94968 with re-fitted thresholds**. The gain over the best single model is modest (+0.0005) — with error correlation at 0.90, there wasn't much disagreement to harvest.

## 4. Final selection

Kaggle scores two chosen submissions on the private LB; the winner is the better one. Strategy: **don't put both eggs in one basket.**

- **Final A (best CV):** the blend + thresholds — highest OOF, most machinery.
- **Final B (robustness hedge):** seed-averaged tuned LightGBM + thresholds — one family, fewer fitted meta-parameters (no blend weights), so less surface area for CV-overfit. If the blend's edge is real it wins; if its edge was fitted noise, B catches the fall.

This hedge logic matters because our CV↔LB gap analysis (notebook 03) showed ±0.003 of sampling noise — comparable to the CV differences between our final candidates.

In [9]:
def make_submission(test_probs, mult, name):
    labels = [CLASSES[i] for i in (test_probs * np.array([1.0, *mult])).argmax(1)]
    sub = pd.DataFrame({'id': test['id'], TARGET: labels})
    sub.to_csv(f'../submissions/{name}.csv', index=False)
    print(f"{name}: {dict(sub[TARGET].value_counts(normalize=True).round(4))}")

make_submission(testp_blend, (ab, bb), '04-final-blend')
a2, b2, cv_robust = tune_thresholds(oof_seedavg, y)
print(f"robust candidate OOF: {cv_robust:.5f} (fit={a2:.2f}, unhealthy={b2:.2f})")
make_submission(testp_seedavg, (a2, b2), '04-robust')

np.save('../artifacts/oof_blend.npy', oof_blend)
np.save('../artifacts/testp_blend.npy', testp_blend)

summary = pd.DataFrame({'OOF bal. acc.': {
    'nb02 baseline': BASE_CV, 'tuned LGBM': cv_tuned, 'tuned + thresholds': cv_thresh,
    'seed-avg (3)': balanced_accuracy_score(y, oof_seedavg.argmax(1)),
    'xgboost': cv_xgb, 'catboost': cv_cat,
    'blend (argmax)': cv_blend_argmax, 'blend + thresholds': cv_blend,
    'robust: seed-avg + thresholds': cv_robust}}).round(5)
summary

04-final-blend: {'at-risk': np.float64(0.8098), 'unhealthy': np.float64(0.1164), 'fit': np.float64(0.0738)}


robust candidate OOF: 0.94961 (fit=2.00, unhealthy=1.99)


04-robust: {'at-risk': np.float64(0.8096), 'unhealthy': np.float64(0.1174), 'fit': np.float64(0.073)}


,OOF bal. acc.
nb02 baseline,0.94894
tuned LGBM,0.94821
tuned + thresholds,0.94961
seed-avg (3),0.94831
xgboost,0.93486
catboost,0.94923
blend (argmax),0.94950
blend + thresholds,0.94968
robust: seed-avg + thresholds,0.94961


## Results

| submission | CV (thresholded) | public LB | CV − LB |
|---|---|---|---|
| 04-final-blend (0.3 LGBM + 0.7 CatBoost + thresholds) | 0.94968 | **0.94946** | +0.0002 |
| 04-robust (seed-avg LGBM + thresholds) | 0.94961 | 0.94918 | +0.0004 |

Both landed inside the ±0.003 noise band our gap analysis predicted; the blend is our best public score. (Kaggle's default final-selection takes the two best public scores — currently the blend and notebook 03's E4. Given that public differences of 0.0001–0.0003 are noise, a defensible manual choice is blend + robust, hedging across *methods* rather than across two nearly-identical LightGBMs.)

## The arc, in five numbers

| lever | score | Δ |
|---|---|---|
| majority baseline (metric probe) | 0.33333 | — |
| + a GBM with default loss | 0.87542 | +0.542 |
| + **align loss with metric** (class weights) | 0.94894 | +0.074 |
| + features (5 experiments, 0 keepers) | 0.94894 | +0.000 |
| + tuning (defaults were fine) | 0.94894 | +0.000 |
| + **align decision rule with metric** (thresholds) | 0.94961 | +0.0007 |
| + cross-family blend | **0.94968** | +0.0001 |

The pattern is unmistakable: **almost the entire score came from alignment — objective ↔ metric ↔ decision rule — not from sophistication.** The two zero-rows aren't wasted work: each is a measured fact that redirected effort (features are exhausted; defaults are fine), bought for a few CV runs, and the tuned model's better-ranked probabilities quietly enabled the threshold gain. The remaining ~0.006 to the leaderboard top likely lives in neural-network ensemble members, stacking, and brute-force compute — steeply diminishing returns per unit of effort.